# sensor-LST benchmark — shard `u00` (late-update rate u = 0%)

Delta Lake 3.2.1 vs Apache Iceberg 1.6.1 vs Apache Hudi 0.15.0 (+ plain Parquet) on Spark 3.5.3, workload built from
TSM-Bench d1. This notebook runs every format x {CoW, MoR} x compaction cycle for **one** u value, 3 repetitions,
formats interleaved in random order inside each round, one fresh JVM per run.

**Kaggle:** Settings → Internet **On**, Accelerator **None** (CPU, 4 cores / 30 GB). Then *Save Version → Save & Run All
(Commit)*. It runs headless (≤ 12 h); results land in the Output tab as `results_u00.zip`.
**Colab:** Runtime → Run all, keep the tab open. Results are written to Google Drive (`MyDrive/sensorlst`) after every
run, so if the session disconnects just Run all again: finished runs are skipped.

In [ ]:
U = 0.0            # this shard
REPS = 3
BUDGET_HOURS = 10.5  # stop starting new runs after this; finished runs are kept
SCALE = "small"
USE_DRIVE = True   # Colab only: write results to MyDrive/sensorlst as they finish (survives disconnects)

In [ ]:
import os, subprocess, sys, shutil, glob, time, socket
try:
    socket.getaddrinfo("pypi.org", 443)
except OSError:
    raise RuntimeError("NO INTERNET in this notebook. Kaggle: Settings -> Internet -> On (needs a phone-verified "
                       "account), then run again / Save Version again.") from None
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
SCRATCH = "/tmp/lh"                      # tables, staging, ivy cache: not kept as output
os.makedirs(SCRATCH, exist_ok=True); os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
RESULTS = f"{WORK}/results"
if USE_DRIVE and not KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/sensorlst/results"
os.makedirs(RESULTS, exist_ok=True)
print("results ->", RESULTS)
CORES = os.cpu_count()
MEM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
DRIVER_MEM = f"{max(2, int(MEM_GB * 0.6))}g"
print(f"cores={CORES} mem={MEM_GB:.1f}GB driver={DRIVER_MEM} kaggle={KAGGLE} notebook-python={sys.version.split()[0]}")
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-11-jdk-headless > /dev/null", shell=True, check=True)
jh = subprocess.run("dirname $(dirname $(readlink -f $(which java)))", shell=True, capture_output=True, text=True).stdout.strip()
os.environ["JAVA_HOME"] = jh
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
# Tested: PySpark 3.5.3 + Delta/Iceberg/Hudi run correctly on Python 3.13 and Java 21 (Kaggle's defaults).
PY = sys.executable
subprocess.run([PY, "-m", "pip", "install", "-q", "pyspark==3.5.3", "delta-spark==3.2.1", "pyarrow", "tabulate"], check=True)
print(subprocess.run([PY, "-c", "import sys, pyspark; print('python', sys.version.split()[0], 'pyspark', pyspark.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

## Code (embedded copy of the `sensorlst` package)

In [ ]:
# Writes the package next to the notebook (absolute paths, no cell magics).
FILES = {'__init__.py': '"""sensor-LST: update-heavy IoT workload for Lakehouse table formats."""\n', 'analyze.py': '"""Aggregate run JSONs into tables and paper figures.\n\n    python -m sensorlst.analyze results/runs --out results/report\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport json\nimport os\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\n\n# Fixed categorical order (validated reference palette, slots 1-4); mode is a secondary encoding (line style / hatch).\nCOLORS = {"delta": "#2a78d6", "iceberg": "#eb6834", "hudi": "#1baf7a", "parquet": "#eda100"}\nSTYLE = {"cow": "-", "mor": "--"}\nHATCH = {"cow": "", "mor": "///"}\nFMT_ORDER = ["delta", "iceberg", "hudi", "parquet"]\nQUERIES = ["Q1_window_agg", "Q2_topk", "Q3_join", "Q4_range", "Q5_scan"]\nINK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"\n\nplt.rcParams.update({\n    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7,\n    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,\n    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False,\n    "axes.spines.right": False, "lines.linewidth": 1.6, "lines.markersize": 4, "figure.dpi": 150,\n})\n\n\ndef label(fmt, mode):\n    if fmt == "parquet":\n        return "Parquet (rewrite)"\n    m = {"delta": {"cow": "CoW", "mor": "DV"}}.get(fmt, {"cow": "CoW", "mor": "MoR"})[mode]\n    return f"{fmt.capitalize()} {m}"\n\n\ndef load(run_dir: str):\n    runs, cps, ops = [], [], []\n    for p in sorted(glob.glob(os.path.join(run_dir, "*.json"))):\n        r = json.load(open(p))\n        c = r["config"]\n        key = {"run_id": r["run_id"], "fmt": c["fmt"], "mode": c["mode"], "u": c["u"],\n               "c": c["compact_every"], "rep": c["rep"]}\n        last = r["checkpoints"][-1]\n        q_all = sum(sum(x[1] for x in cp["queries"].values()) for cp in r["checkpoints"])\n        runs.append({**key, "correct": r["correct"], "write_s": r["total_write_s"],\n                     "compact_s": r["total_compact_s"], "query_s": q_all,\n                     "total_s": r["total_write_s"] + r["total_compact_s"] + q_all,\n                     "wa": r["write_amplification"], "logical_mb": r["logical_bytes"] / 2**20,\n                     "live_files": last["data_files"], "delta_files": last["delete_files"],\n                     "meta_kb": last["meta_bytes_on_disk"] / 1024, "meta_files": last["meta_files_on_disk"],\n                     "n_versions": last.get("n_versions"),\n                     "final_q_s": sum(v[1] for v in last["queries"].values()),\n                     **{f"final_{q}": last["queries"][q][1] for q in QUERIES},\n                     "tt_back_s": last.get("time_travel", {}).get("tt_back", {}).get("seconds"),\n                     "tt_first_s": last.get("time_travel", {}).get("tt_first", {}).get("seconds"),\n                     "jvm_s": r["jvm_start_s"], "cpus": r["host"]["cpus"]})\n        for cp in r["checkpoints"]:\n            cps.append({**key, "batch": cp["batch"], "q_warm_s": sum(v[1] for v in cp["queries"].values()),\n                        "q_cold_s": sum(v[0] for v in cp["queries"].values()),\n                        "live_files": cp["data_files"], "delta_files": cp["delete_files"],\n                        "meta_kb": cp["meta_bytes_on_disk"] / 1024, "n_versions": cp.get("n_versions"),\n                        "tt_back_s": cp.get("time_travel", {}).get("tt_back", {}).get("seconds"),\n                        "tt_first_s": cp.get("time_travel", {}).get("tt_first", {}).get("seconds"),\n                        "correct": cp["correct"]})\n        for o in r["ops"]:\n            ops.append({**key, **o})\n    return pd.DataFrame(runs), pd.DataFrame(cps), pd.DataFrame(ops)\n\n\ndef p95(x):\n    return np.percentile(x, 95)\n\n\ndef cv(x):\n    return float(np.std(x, ddof=1) / np.mean(x)) if len(x) > 1 and np.mean(x) else np.nan\n\n\ndef summarise(runs: pd.DataFrame) -> pd.DataFrame:\n    g = runs.groupby(["fmt", "mode", "u", "c"])\n    s = g.agg(n=("rep", "count"), all_correct=("correct", "all"),\n              write_s=("write_s", "median"), compact_s=("compact_s", "median"),\n              query_s=("query_s", "median"), total_s=("total_s", "median"),\n              total_s_p95=("total_s", p95), total_cv=("total_s", cv),\n              final_q_s=("final_q_s", "median"), final_q_p95=("final_q_s", p95),\n              wa=("wa", "median"), live_files=("live_files", "median"), delta_files=("delta_files", "median"),\n              meta_kb=("meta_kb", "median"), n_versions=("n_versions", "median"),\n              tt_back_s=("tt_back_s", "median"), tt_first_s=("tt_first_s", "median"))\n    return s.reset_index()\n\n\ndef degradation(cps: pd.DataFrame) -> pd.DataFrame:\n    """LST-Bench-style degradation rate: relative growth of warm query time from first to last checkpoint."""\n    def rate(df):\n        df = df.sort_values("batch")\n        return (df.q_warm_s.iloc[-1] - df.q_warm_s.iloc[0]) / df.q_warm_s.iloc[0]\n    d = cps.groupby(["fmt", "mode", "u", "c", "rep"]).apply(rate, include_groups=False).rename("degradation")\n    return d.reset_index().groupby(["fmt", "mode", "u", "c"]).degradation.median().reset_index()\n\n\ndef _series(s):\n    for fmt in FMT_ORDER:\n        for mode in ("cow", "mor"):\n            d = s[(s.fmt == fmt) & (s["mode"] == mode)]\n            if len(d):\n                yield fmt, mode, d\n\n\ndef _legend(fig, axes, ncol, y):\n    """One legend for the figure, from every subplot, in the fixed series order."""\n    seen = {}\n    for ax in np.ravel(axes):\n        for h, l in zip(*ax.get_legend_handles_labels()):\n            seen.setdefault(l, h)\n    order = [label(f, m) for f in FMT_ORDER for m in ("cow", "mor")]\n    labels = [l for l in order if l in seen]\n    fig.legend([seen[l] for l in labels], labels, loc="lower center", ncol=ncol, frameon=False,\n               bbox_to_anchor=(0.5, y))\n\n\ndef fig_rq1(s, out):\n    cs = sorted(s.c.unique())\n    fig, axes = plt.subplots(1, len(cs), figsize=(2.4 * len(cs) + 0.4, 2.4), sharey=True, squeeze=False)\n    for ax, c in zip(axes[0], cs):\n        sc = s[(s.c == c) | ((s.fmt == "parquet") & (s.c == 0))]\n        for fmt, mode, d in _series(sc):\n            d = d.sort_values("u")\n            ax.plot(d.u * 100, d.total_s, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n        ax.set_title("no compaction" if c == 0 else f"compaction every {c} batches")\n        ax.set_xlabel("late-update rate u (%)")\n    axes[0][0].set_ylabel("total cost (s): ingest + compaction + queries")\n    _legend(fig, axes, 4, -0.16)\n    fig.tight_layout()\n    fig.savefig(os.path.join(out, "fig_rq1_total_cost.png"), bbox_inches="tight")\n    plt.close(fig)\n\n\ndef fig_write_amp(s, out):\n    sc = s[s.c == 0]\n    us = sorted(sc.u.unique())\n    groups = [(f, m) for f, m, _ in _series(sc)]\n    fig, ax = plt.subplots(figsize=(5.2, 2.3))\n    w = 0.8 / max(1, len(groups))\n    for i, (fmt, mode) in enumerate(groups):\n        d = sc[(sc.fmt == fmt) & (sc["mode"] == mode)].set_index("u").reindex(us)\n        ax.bar(np.arange(len(us)) + i * w, d.wa, w * 0.92, color=COLORS[fmt], hatch=HATCH[mode],\n               edgecolor="white", linewidth=0.5, label=label(fmt, mode))\n    ax.set_xticks(np.arange(len(us)) + w * (len(groups) - 1) / 2, [f"u={u * 100:g}%" for u in us])\n    ax.set_ylabel("write amplification\\n(bytes written / logical bytes)")\n    ax.legend(ncol=4, frameon=False, loc="upper left", bbox_to_anchor=(0, 1.25))\n    fig.tight_layout()\n    fig.savefig(os.path.join(out, "fig_write_amplification.png"), bbox_inches="tight")\n    plt.close(fig)\n\n\ndef fig_rq2(s, cps, out):\n    u = s.u.max()\n    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.4))\n    for fmt, mode, d in _series(s[s.u == u]):\n        d = d.sort_values("c")\n        x = [str(c) if c else "never" for c in d.c]\n        axes[0].plot(x, d.final_q_s, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n        axes[1].plot(x, d.live_files + d.delta_files, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n    axes[0].set_ylabel("Q1-Q5 latency at end (s, warm)")\n    axes[1].set_ylabel("live files (data + delete/log)")\n    for ax in axes:\n        ax.set_xlabel("compaction every k batches")\n    axes[0].set_title(f"read cost vs compaction (u={u * 100:g}%)")\n    axes[1].set_title("small-file count at end")\n    _legend(fig, axes, 4, -0.2)\n    fig.tight_layout()\n    fig.savefig(os.path.join(out, "fig_rq2_compaction.png"), bbox_inches="tight")\n    plt.close(fig)\n\n    # read-latency trajectory without compaction (degradation)\n    d0 = cps[(cps.c == 0) & (cps.u == u)].groupby(["fmt", "mode", "batch"]).q_warm_s.median().reset_index()\n    fig, ax = plt.subplots(figsize=(3.4, 2.4))\n    for fmt, mode, d in _series(d0):\n        ax.plot(d.batch, d.q_warm_s, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n    ax.set_xlabel("micro-batches committed")\n    ax.set_ylabel("Q1-Q5 latency (s, warm)")\n    ax.set_title(f"read degradation, no compaction (u={u * 100:g}%)")\n    ax.legend(frameon=False, ncol=2)\n    fig.tight_layout()\n    fig.savefig(os.path.join(out, "fig_rq2_degradation.png"), bbox_inches="tight")\n    plt.close(fig)\n\n\ndef fig_rq3(cps, out):\n    d = cps[(cps.c == 0) & cps.n_versions.notna()]\n    u = d.u.max()\n    d = d[d.u == u].groupby(["fmt", "mode", "batch"])[["meta_kb", "n_versions", "tt_back_s", "tt_first_s"]] \\\n        .median().reset_index()\n    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.4))\n    for fmt, mode, g in _series(d):\n        axes[0].plot(g.n_versions, g.meta_kb / 1024, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n        axes[1].plot(g.n_versions, g.tt_back_s, STYLE[mode], color=COLORS[fmt], marker="o", label=label(fmt, mode))\n    axes[0].set_xlabel("commits / snapshots")\n    axes[0].set_ylabel("metadata on disk (MB)")\n    axes[1].set_xlabel("commits / snapshots")\n    axes[1].set_ylabel("time travel to T-5 (s)")\n    axes[0].set_title(f"metadata growth (u={u * 100:g}%, no compaction)")\n    axes[1].set_title("time-travel latency")\n    _legend(fig, axes, 3, -0.2)\n    fig.tight_layout()\n    fig.savefig(os.path.join(out, "fig_rq3_metadata.png"), bbox_inches="tight")\n    plt.close(fig)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("runs", nargs="+")\n    ap.add_argument("--out", default="results/report")\n    args = ap.parse_args()\n    os.makedirs(args.out, exist_ok=True)\n    parts = [load(r) for r in args.runs]\n    runs, cps, ops = (pd.concat([p[i] for p in parts], ignore_index=True) for i in range(3))\n    runs.to_csv(os.path.join(args.out, "runs.csv"), index=False)\n    cps.to_csv(os.path.join(args.out, "checkpoints.csv"), index=False)\n    ops.to_csv(os.path.join(args.out, "ops.csv"), index=False)\n    s = summarise(runs).merge(degradation(cps), on=["fmt", "mode", "u", "c"], how="left")\n    s.to_csv(os.path.join(args.out, "summary.csv"), index=False)\n    op_s = ops.groupby(["fmt", "mode", "u", "op"]).seconds.median().unstack("op").reset_index()\n    op_s.to_csv(os.path.join(args.out, "op_latency_median.csv"), index=False)\n\n    fig_rq1(s, args.out)\n    fig_write_amp(s, args.out)\n    fig_rq2(s, cps, args.out)\n    fig_rq3(cps, args.out)\n\n    with open(os.path.join(args.out, "summary.md"), "w", encoding="utf-8") as f:\n        f.write(f"runs: {len(runs)}, all correct: {bool(runs.correct.all())}\\n\\n")\n        f.write(s.round(3).to_markdown(index=False))\n    print(f"{len(runs)} runs, correct={runs.correct.all()} -> {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'config.py': '"""Experiment configuration: workload scales, run configs and the experiment matrix."""\nfrom __future__ import annotations\n\nimport itertools\nimport json\nfrom dataclasses import asdict, dataclass, field\n\n# Pinned versions (Spark 3.5 line). Changing any of these invalidates comparisons.\nVERSIONS = {\n    "spark": "3.5.3",\n    "delta": "3.2.1",\n    "iceberg": "1.6.1",\n    "hudi": "0.15.0",\n}\n\nPACKAGES = {\n    "delta": f"io.delta:delta-spark_2.12:{VERSIONS[\'delta\']}",\n    "iceberg": f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{VERSIONS[\'iceberg\']}",\n    "hudi": f"org.apache.hudi:hudi-spark3.5-bundle_2.12:{VERSIONS[\'hudi\']}",\n    "parquet": None,\n}\n\nFORMATS = ("delta", "iceberg", "hudi", "parquet")\nMODES = ("cow", "mor")\nU_LEVELS = (0.0, 0.01, 0.05, 0.20)\n\n\n@dataclass(frozen=True)\nclass Scale:\n    """Shape of the sensor-LST workload. Event time advances in fixed steps of `interval_s`."""\n\n    name: str\n    n_stations: int\n    n_sensors: int\n    base_hours: float          # history bulk-loaded before the first micro-batch\n    batch_minutes: int         # event time covered by one micro-batch\n    n_batches: int\n    late_window_hours: float   # late corrections hit keys from this window before the batch\n    delete_every: int          # a station-day delete every k batches\n    checkpoint_every: int      # queries + metadata stats every k batches\n    compaction_levels: tuple   # compaction every k batches; 0 = never\n    interval_s: int = 10\n    start: str = "2019-03-01T00:00:00"\n\n    def rows_per_batch(self) -> int:\n        return self.n_stations * self.batch_minutes * 60 // self.interval_s\n\n\nSCALES = {\n    # seconds on a laptop; used by tests and local smoke runs\n    "smoke": Scale("smoke", 4, 5, 6, 30, 6, 2, 3, 3, (0, 2)),\n    # main experiment: 10 stations x 20 sensors of TSM-Bench d1, 7 days history + 10 days of 6-h batches\n    "small": Scale("small", 10, 20, 24 * 7, 360, 40, 72, 10, 10, (0, 5, 20)),\n}\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    fmt: str\n    mode: str            # cow | mor (Delta: mor = deletion vectors)\n    u: float             # late-update fraction relative to batch size\n    compact_every: int   # 0 = never\n    rep: int\n    scale: str\n    seed: int = 42\n    tags: dict = field(default_factory=dict, compare=False, hash=False)\n\n    @property\n    def run_id(self) -> str:\n        return (f"{self.scale}_{self.fmt}_{self.mode}_u{int(round(self.u * 100)):02d}"\n                f"_c{self.compact_every:02d}_r{self.rep}")\n\n    def to_json(self) -> str:\n        return json.dumps(asdict(self))\n\n    @staticmethod\n    def from_json(s: str) -> "RunConfig":\n        return RunConfig(**json.loads(s))\n\n\ndef matrix(scale: str, u_levels=U_LEVELS, formats=FORMATS, reps: int = 3) -> list[RunConfig]:\n    """Full factorial: table formats x {cow,mor} x u x compaction cycle; plain Parquet as cow/no-compaction baseline."""\n    sc = SCALES[scale]\n    out = []\n    for rep, u in itertools.product(range(1, reps + 1), u_levels):\n        for fmt in formats:\n            if fmt == "parquet":\n                out.append(RunConfig("parquet", "cow", u, 0, rep, scale))\n                continue\n            for mode, c in itertools.product(MODES, sc.compaction_levels):\n                out.append(RunConfig(fmt, mode, u, c, rep, scale))\n    return out\n', 'formats.py': '"""One adapter per storage layout. Every adapter exposes the same operations so the runner is format-agnostic.\n\nDesign choices that keep the comparison controlled (see README "Fairness"):\n  * one Spark engine, one JVM per run, same staged input;\n  * snappy Parquet everywhere (Iceberg defaults to zstd, Hudi to gzip);\n  * old file versions are never cleaned during a run (no VACUUM / expire / Hudi cleaner), so bytes\n    on disk = bytes written, and every version stays reachable for time travel;\n  * partitioned by `day`; MERGE/DELETE carry an explicit `day IN (...)` predicate for pruning.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport shutil\n\nfrom pyspark.sql import DataFrame, SparkSession\nfrom pyspark.sql import functions as F\nfrom pyspark.sql.window import Window\n\nfrom .config import PACKAGES\n\nDATA_COLS_EXCLUDE = re.compile(r"^_hoodie_")\n\n\ndef build_spark(fmt: str, workdir: str, cores: int = 4, driver_mem: str = "8g", ivy_dir: str | None = None) -> SparkSession:\n    b = (SparkSession.builder.master(f"local[{cores}]").appName(f"sensor-lst-{fmt}")\n         .config("spark.driver.memory", driver_mem)\n         .config("spark.sql.shuffle.partitions", str(cores * 2))\n         .config("spark.sql.session.timeZone", "UTC")\n         .config("spark.driver.extraJavaOptions", "-Duser.timezone=UTC")\n         .config("spark.ui.enabled", "false")\n         .config("spark.ui.showConsoleProgress", "false")\n         .config("spark.sql.parquet.compression.codec", "snappy")\n         .config("spark.local.dir", os.path.join(workdir, "spark-local")))\n    if ivy_dir:\n        b = b.config("spark.jars.ivy", ivy_dir)\n    if PACKAGES[fmt]:\n        b = b.config("spark.jars.packages", PACKAGES[fmt])\n    if fmt == "delta":\n        b = (b.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\n              .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))\n    elif fmt == "iceberg":\n        b = (b.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")\n              .config("spark.sql.catalog.ice", "org.apache.iceberg.spark.SparkCatalog")\n              .config("spark.sql.catalog.ice.type", "hadoop")\n              .config("spark.sql.catalog.ice.warehouse", _uri(os.path.join(workdir, "iceberg_wh"))))\n    elif fmt == "hudi":\n        b = (b.config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")\n              .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")\n              .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\n              .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar"))\n    spark = b.getOrCreate()\n    spark.sparkContext.setLogLevel("ERROR")\n    return spark\n\n\ndef _uri(path: str) -> str:\n    return "file:///" + os.path.abspath(path).replace("\\\\", "/").lstrip("/")\n\n\ndef _sql_path(path: str) -> str:\n    return os.path.abspath(path).replace("\\\\", "/")\n\n\ndef _in_list(values) -> str:\n    return ", ".join(f"\'{v}\'" for v in sorted(set(values)))\n\n\nclass Adapter:\n    """Base adapter. `root` is the directory that holds the table and nothing else."""\n\n    meta_dirs: tuple = ()\n    supports_time_travel = True\n\n    def __init__(self, spark: SparkSession, workdir: str, mode: str):\n        self.spark, self.workdir, self.mode = spark, workdir, mode\n        self.root = os.path.abspath(os.path.join(workdir, "table"))\n\n    # --- operations ---------------------------------------------------------------------------\n    def create(self, df: DataFrame) -> None: raise NotImplementedError\n    def append(self, df: DataFrame) -> None: raise NotImplementedError\n    def upsert(self, df: DataFrame) -> None: raise NotImplementedError\n    def delete(self, station: str, day: str) -> None: raise NotImplementedError\n    def compact(self) -> None: raise NotImplementedError\n    def read(self) -> DataFrame: raise NotImplementedError\n    def versions(self) -> list: return []\n    def read_version(self, ref) -> DataFrame: raise NotImplementedError\n\n    # --- storage accounting -------------------------------------------------------------------\n    def disk_files(self) -> dict[str, int]:\n        """All files under the table root -> size."""\n        out = {}\n        for dp, _, fs in os.walk(self.root):\n            for f in fs:\n                p = os.path.join(dp, f)\n                try:\n                    out[p] = os.path.getsize(p)\n                except OSError:\n                    pass\n        return out\n\n    def classify(self, path: str) -> str:\n        """\'meta\' | \'data\' | \'ignore\'. Hadoop .crc sidecars exist only on the local FS (not on S3) -> ignored."""\n        rel = os.path.relpath(path, self.root).replace("\\\\", "/")\n        name = rel.rsplit("/", 1)[-1]\n        if name.endswith(".crc") or name == "_SUCCESS":\n            return "ignore"\n        if any(rel == d or rel.startswith(d + "/") for d in self.meta_dirs):\n            return "meta"\n        return "data"\n\n    def live_files(self) -> dict: raise NotImplementedError\n\n\n# ---------------------------------------------------------------------------------------------\nclass DeltaAdapter(Adapter):\n    meta_dirs = ("_delta_log",)\n\n    def __init__(self, spark, workdir, mode):\n        super().__init__(spark, workdir, mode)\n        dv = "true" if mode == "mor" else "false"\n        spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", dv)\n        self.t = f"delta.`{_sql_path(self.root)}`"\n\n    def create(self, df):\n        df.write.format("delta").partitionBy("day").mode("overwrite").save(self.root)\n\n    def append(self, df):\n        df.write.format("delta").mode("append").save(self.root)\n\n    def upsert(self, df):\n        days = [r.day for r in df.select("day").distinct().collect()]\n        df.createOrReplaceTempView("src")\n        self.spark.sql(f"""MERGE INTO {self.t} t USING src s\n            ON t.day IN ({_in_list(days)}) AND t.day = s.day AND t.id_station = s.id_station AND t.time = s.time\n            WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")\n\n    def delete(self, station, day):\n        self.spark.sql(f"DELETE FROM {self.t} WHERE day = \'{day}\' AND id_station = \'{station}\'")\n\n    def compact(self):\n        if self.mode == "mor":  # materialise deletion vectors, then bin-pack\n            self.spark.sql(f"REORG TABLE {self.t} APPLY (PURGE)")\n        self.spark.sql(f"OPTIMIZE {self.t}")\n\n    def read(self):\n        return self.spark.read.format("delta").load(self.root)\n\n    def versions(self):\n        return [r.version for r in self.spark.sql(f"DESCRIBE HISTORY {self.t}").select("version").orderBy("version").collect()]\n\n    def read_version(self, ref):\n        return self.spark.read.format("delta").option("versionAsOf", ref).load(self.root)\n\n    def live_files(self):\n        """Replay the JSON commit log (never cleaned during a run): live files and live deletion vectors."""\n        log = os.path.join(self.root, "_delta_log")\n        live = {}\n        for name in sorted(f for f in os.listdir(log) if re.fullmatch(r"\\d{20}\\.json", f)):\n            with open(os.path.join(log, name)) as fh:\n                for line in fh:\n                    a = json.loads(line)\n                    # a DV change removes and re-adds the same path in one commit: key on (path, DV)\n                    if "add" in a:\n                        dv = a["add"].get("deletionVector")\n                        live[(a["add"]["path"], json.dumps(dv, sort_keys=True))] = (a["add"]["size"], dv)\n                    elif "remove" in a:\n                        dv = a["remove"].get("deletionVector")\n                        live.pop((a["remove"]["path"], json.dumps(dv, sort_keys=True)), None)\n        dvs = [dv for _, dv in live.values() if dv]\n        return {"data_files": len(live), "data_bytes": sum(s for s, _ in live.values()),\n                "delete_files": len(dvs), "delete_bytes": sum(dv.get("sizeInBytes", 0) for dv in dvs)}\n\n\n# ---------------------------------------------------------------------------------------------\nclass IcebergAdapter(Adapter):\n    t = "ice.db.t"\n\n    def __init__(self, spark, workdir, mode):\n        super().__init__(spark, workdir, mode)\n        self.root = os.path.abspath(os.path.join(workdir, "iceberg_wh", "db", "t"))\n        self.meta_dirs = ("metadata",)\n\n    def create(self, df):\n        m = "merge-on-read" if self.mode == "mor" else "copy-on-write"\n        (df.writeTo(self.t).partitionedBy(F.col("day"))\n           .tableProperty("format-version", "2")\n           .tableProperty("write.parquet.compression-codec", "snappy")\n           .tableProperty("write.delete.mode", m).tableProperty("write.update.mode", m)\n           .tableProperty("write.merge.mode", m)\n           .createOrReplace())\n\n    def append(self, df):\n        df.writeTo(self.t).append()\n\n    def upsert(self, df):\n        days = [r.day for r in df.select("day").distinct().collect()]\n        df.createOrReplaceTempView("src")\n        self.spark.sql(f"""MERGE INTO {self.t} t USING src s\n            ON t.day IN ({_in_list(days)}) AND t.day = s.day AND t.id_station = s.id_station AND t.time = s.time\n            WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")\n\n    def delete(self, station, day):\n        self.spark.sql(f"DELETE FROM {self.t} WHERE day = \'{day}\' AND id_station = \'{station}\'")\n\n    def compact(self):\n        self.spark.sql("CALL ice.system.rewrite_data_files(table => \'db.t\', "\n                       "options => map(\'min-input-files\', \'2\', \'delete-file-threshold\', \'1\'))")\n        if self.mode == "mor":\n            self.spark.sql("CALL ice.system.rewrite_position_delete_files(table => \'db.t\')")\n\n    def read(self):\n        return self.spark.table(self.t)\n\n    def versions(self):\n        return [r.snapshot_id for r in\n                self.spark.sql(f"SELECT snapshot_id FROM {self.t}.snapshots ORDER BY committed_at").collect()]\n\n    def read_version(self, ref):\n        return self.spark.sql(f"SELECT * FROM {self.t} VERSION AS OF {ref}")\n\n    def live_files(self):\n        rows = self.spark.sql(f"SELECT content, count(*) n, sum(file_size_in_bytes) b FROM {self.t}.files "\n                              f"GROUP BY content").collect()\n        by = {r.content: (r.n, r.b or 0) for r in rows}\n        dn = sum(v[0] for k, v in by.items() if k != 0)\n        db = sum(v[1] for k, v in by.items() if k != 0)\n        return {"data_files": by.get(0, (0, 0))[0], "data_bytes": by.get(0, (0, 0))[1],\n                "delete_files": dn, "delete_bytes": db}\n\n\n# ---------------------------------------------------------------------------------------------\n_HUDI_BASE = re.compile(r"^(?P<fid>[0-9a-f-]+-\\d+)_[^_]+_(?P<ts>\\d+)\\.parquet$")\n_HUDI_LOG = re.compile(r"^\\.(?P<fid>[0-9a-f-]+-\\d+)_(?P<ts>\\d+)\\.log\\.\\d+_.+$")\n_HUDI_DONE = re.compile(r"^(\\d+)\\.(commit|deltacommit|replacecommit)$")\n\n\nclass HudiAdapter(Adapter):\n    meta_dirs = (".hoodie",)\n\n    def __init__(self, spark, workdir, mode):\n        super().__init__(spark, workdir, mode)\n        self.opts = {\n            "hoodie.table.name": "t",\n            "hoodie.datasource.write.table.type": "MERGE_ON_READ" if mode == "mor" else "COPY_ON_WRITE",\n            # Hudi renders timestamp key fields differently on its bulk_insert (row) and upsert (Avro) paths,\n            # which silently duplicates records; key on a deterministic string column instead.\n            "hoodie.datasource.write.recordkey.field": "hkey",\n            "hoodie.datasource.write.partitionpath.field": "day",\n            "hoodie.datasource.write.precombine.field": "ingest_batch",\n            "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.SimpleKeyGenerator",\n            "hoodie.datasource.write.hive_style_partitioning": "true",\n            "hoodie.parquet.compression.codec": "snappy",\n            "hoodie.clean.automatic": "false",\n            "hoodie.compact.inline": "false",\n            "hoodie.clustering.inline": "false",\n        }\n        # Default compaction scheduling waits for 5 delta commits; the experiment controls the cadence itself.\n        spark.conf.set("hoodie.compact.inline.max.delta.commits", "1")\n\n    def _write(self, df, op, mode="append"):\n        if "hkey" not in df.columns:\n            df = df.withColumn("hkey", F.concat_ws("|", "id_station", F.unix_micros("time").cast("string")))\n        (df.write.format("hudi").options(**self.opts).option("hoodie.datasource.write.operation", op)\n           .mode(mode).save(_uri(self.root)))\n\n    def create(self, df):\n        self._write(df, "bulk_insert", "overwrite")\n\n    def append(self, df):\n        self._write(df, "insert")\n\n    def upsert(self, df):\n        self._write(df, "upsert")\n\n    def delete(self, station, day):\n        doomed = self.read().where(f"day = \'{day}\' AND id_station = \'{station}\'")\n        self._write(doomed.drop(*[c for c in doomed.columns if c.startswith("_hoodie_")]), "delete")\n\n    def compact(self):\n        p = _uri(self.root)\n        if self.mode == "mor":\n            self.spark.sql(f"CALL run_compaction(op => \'scheduleAndExecute\', path => \'{p}\')").collect()\n        else:\n            self.spark.sql(f"CALL run_clustering(path => \'{p}\')").collect()\n\n    def read(self):\n        return self.spark.read.format("hudi").load(_uri(self.root))\n\n    def versions(self):\n        hd = os.path.join(self.root, ".hoodie")\n        return sorted(m.group(1) for f in os.listdir(hd) if (m := _HUDI_DONE.match(f)))\n\n    def read_version(self, ref):\n        return self.spark.read.format("hudi").option("as.of.instant", ref).load(_uri(self.root))\n\n    def live_files(self):\n        """Latest file slice per file group, minus file groups replaced by clustering."""\n        hd = os.path.join(self.root, ".hoodie")\n        done = set(self.versions())\n        replaced = set()\n        for f in os.listdir(hd):\n            if f.endswith(".replacecommit") and f.split(".")[0] in done:\n                with open(os.path.join(hd, f)) as fh:\n                    meta = json.load(fh)\n                for ids in (meta.get("partitionToReplaceFileIds") or {}).values():\n                    replaced.update(ids)\n        groups: dict[str, dict] = {}\n        for p, size in self.disk_files().items():\n            if os.sep + ".hoodie" in p or "/.hoodie" in p:\n                continue\n            name = os.path.basename(p)\n            if m := _HUDI_BASE.match(name):\n                if m["ts"] not in done:\n                    continue\n                g = groups.setdefault(m["fid"], {"base": None, "logs": []})\n                if g["base"] is None or m["ts"] > g["base"][0]:\n                    g["base"] = (m["ts"], size)\n            elif m := _HUDI_LOG.match(name):\n                groups.setdefault(m["fid"], {"base": None, "logs": []})["logs"].append((m["ts"], size))\n        out = {"data_files": 0, "data_bytes": 0, "delete_files": 0, "delete_bytes": 0}\n        for fid, g in groups.items():\n            if fid in replaced:\n                continue\n            base_ts = g["base"][0] if g["base"] else ""\n            if g["base"]:\n                out["data_files"] += 1\n                out["data_bytes"] += g["base"][1]\n            logs = [s for ts, s in g["logs"] if ts >= base_ts]\n            out["delete_files"] += len(logs)  # log files = row-level changes not yet compacted\n            out["delete_bytes"] += sum(logs)\n        return out\n\n\n# ---------------------------------------------------------------------------------------------\nclass ParquetAdapter(Adapter):\n    """Plain Parquet baseline: no log, no ACID; row changes rewrite whole day partitions."""\n\n    supports_time_travel = False\n\n    def create(self, df):\n        df.write.partitionBy("day").mode("overwrite").parquet(self.root)\n\n    def append(self, df):\n        df.write.partitionBy("day").mode("append").parquet(self.root)\n\n    def _rewrite_days(self, days, transform):\n        days = sorted(set(days))\n        old = self.read().where(F.col("day").isin(days))\n        new = transform(old)\n        tmp = os.path.join(self.workdir, "parquet_tmp")\n        shutil.rmtree(tmp, ignore_errors=True)\n        new.write.partitionBy("day").parquet(tmp)\n        for d in days:  # swap partition directories (not atomic -- that is the point of the baseline)\n            shutil.rmtree(os.path.join(self.root, f"day={d}"), ignore_errors=True)\n            src = os.path.join(tmp, f"day={d}")\n            if os.path.isdir(src):\n                shutil.move(src, os.path.join(self.root, f"day={d}"))\n        shutil.rmtree(tmp, ignore_errors=True)\n\n    def upsert(self, df):\n        days = [r.day for r in df.select("day").distinct().collect()]\n        cols = df.columns\n\n        def merge(old):\n            w = Window.partitionBy("id_station", "time").orderBy(F.col("ingest_batch").desc())\n            both = old.select(*cols).unionByName(df.select(*cols))\n            return both.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")\n        self._rewrite_days(days, merge)\n\n    def delete(self, station, day):\n        self._rewrite_days([day], lambda old: old.where(F.col("id_station") != station))\n\n    def compact(self):\n        pass\n\n    def read(self):\n        return self.spark.read.parquet(self.root)\n\n    def live_files(self):\n        files = {p: s for p, s in self.disk_files().items() if p.endswith(".parquet")}\n        return {"data_files": len(files), "data_bytes": sum(files.values()), "delete_files": 0, "delete_bytes": 0}\n\n\nADAPTERS = {"delta": DeltaAdapter, "iceberg": IcebergAdapter, "hudi": HudiAdapter, "parquet": ParquetAdapter}\n', 'matrix.py': '"""Run (a shard of) the experiment matrix. Resumable: finished run JSONs are skipped.\n\nRuns are grouped in rounds (rep 1 of every config, then rep 2, ...), and the order inside each round\nis shuffled, so formats are interleaved on the same machine and an expired time budget still leaves\ncomplete rounds.\n\n    python -m sensorlst.matrix --scale small --u 0.05 --source data/d1_small.parquet --out results/runs\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport random\nimport subprocess\nimport sys\nimport time\n\nimport pandas as pd\n\nfrom .config import FORMATS, SCALES, RunConfig, matrix\nfrom .workload import build_plan\n\n\ndef stage(scale: str, u: float, source: str | None, staging_root: str, seed: int) -> str:\n    d = os.path.join(staging_root, f"{scale}_u{int(round(u * 100)):02d}_s{seed}")\n    if not os.path.exists(os.path.join(d, "manifest.json")):\n        src = pd.read_parquet(source) if source else None\n        t = time.time()\n        m = build_plan(scale, u, d, src, seed)\n        print(f"staged {d}: base={m[\'base_rows\']} batches={len(m[\'batches\'])} in {time.time() - t:.0f}s", flush=True)\n    return d\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--scale", default="small", choices=list(SCALES))\n    ap.add_argument("--u", type=float, nargs="+", required=True)\n    ap.add_argument("--formats", default=",".join(FORMATS))\n    ap.add_argument("--reps", type=int, default=3)\n    ap.add_argument("--source", default=None, help="Parquet from prepare_d1; omit for synthetic data")\n    ap.add_argument("--out", default="results/runs")\n    ap.add_argument("--staging", default="staging")\n    ap.add_argument("--workdir", default="work")\n    ap.add_argument("--cores", type=int, default=4)\n    ap.add_argument("--driver-mem", default="8g")\n    ap.add_argument("--ivy", default=None)\n    ap.add_argument("--budget-hours", type=float, default=11.0)\n    ap.add_argument("--timeout-min", type=float, default=60)\n    ap.add_argument("--calibration", action="store_true",\n                    help="add delta/cow/u=0/no-compaction to every round, to compare machines across shards")\n    ap.add_argument("--seed", type=int, default=42)\n    args = ap.parse_args()\n\n    deadline = time.time() + args.budget_hours * 3600\n    os.makedirs(args.out, exist_ok=True)\n    formats = tuple(args.formats.split(","))\n    cfgs = matrix(args.scale, tuple(args.u), formats, args.reps)\n    if args.calibration and 0.0 not in args.u:\n        cfgs += [RunConfig("delta", "cow", 0.0, 0, r, args.scale, tags={"calibration": True})\n                 for r in range(1, args.reps + 1)]\n    stagings = {u: stage(args.scale, u, args.source, args.staging, args.seed) for u in {c.u for c in cfgs}}\n\n    durations = []\n    for rep in range(1, args.reps + 1):\n        rnd = [c for c in cfgs if c.rep == rep]\n        random.Random(args.seed * 1000 + rep).shuffle(rnd)\n        for c in rnd:\n            dst = os.path.join(args.out, f"{c.run_id}.json")\n            if os.path.exists(dst):\n                continue\n            est = (sorted(durations)[len(durations) // 2] if durations else 0)\n            if time.time() + est > deadline:\n                print("time budget exhausted; stopping", flush=True)\n                return\n            t = time.time()\n            cmd = [sys.executable, "-m", "sensorlst.run_one", "--config", c.to_json(),\n                   "--staging", stagings[c.u], "--workdir", os.path.join(args.workdir, c.run_id),\n                   "--out", args.out, "--cores", str(args.cores), "--driver-mem", args.driver_mem]\n            if args.ivy:\n                cmd += ["--ivy", args.ivy]\n            try:\n                p = subprocess.run(cmd, capture_output=True, text=True, timeout=args.timeout_min * 60)\n                tail = (p.stdout + p.stderr).strip().splitlines()[-1:] if p.returncode == 0 else \\\n                       (p.stderr or p.stdout).strip().splitlines()[-3:]\n            except subprocess.TimeoutExpired:\n                with open(os.path.join(args.out, "failures.jsonl"), "a") as f:\n                    f.write(json.dumps({"run_id": c.run_id, "error": "timeout"}) + "\\n")\n                tail = ["TIMEOUT"]\n            durations.append(time.time() - t)\n            print(f"[{time.strftime(\'%H:%M:%S\')}] {c.run_id} {time.time() - t:.0f}s :: {\' | \'.join(tail)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'queries.py': '"""Analytical queries of the sensor-LST workload, run against a temp view `t` of the current snapshot.\n\nThey follow the monitoring use case of TSM-Bench: recent-window aggregation, top-k, join with\nstation metadata, selective range scan, and a full scan that doubles as the correctness checksum.\n"""\nfrom __future__ import annotations\n\nimport pandas as pd\n\n\ndef _ts(x: pd.Timestamp) -> str:\n    return x.strftime("%Y-%m-%d %H:%M:%S")\n\n\ndef _day(x: pd.Timestamp) -> str:\n    return x.strftime("%Y-%m-%d")\n\n\ndef make_queries(event_time_end: str) -> dict[str, str]:\n    end = pd.Timestamp(event_time_end)\n    d1, d3 = end - pd.Timedelta(hours=24), end - pd.Timedelta(hours=72)\n    r0 = end - pd.Timedelta(hours=30)\n    return {\n        # Q1: hourly averages per station over the last 24 h (dashboard refresh)\n        "Q1_window_agg": f"""SELECT id_station, date_trunc(\'HOUR\', time) h, avg(s0), avg(s1), avg(s2), count(*)\n            FROM t WHERE day >= \'{_day(d1)}\' AND time >= TIMESTAMP \'{_ts(d1)}\'\n            GROUP BY id_station, date_trunc(\'HOUR\', time)""",\n        # Q2: top-5 stations by peak value over the last 72 h (alerting)\n        "Q2_topk": f"""SELECT id_station, max(s3) m FROM t\n            WHERE day >= \'{_day(d3)}\' AND time >= TIMESTAMP \'{_ts(d3)}\'\n            GROUP BY id_station ORDER BY m DESC LIMIT 5""",\n        # Q3: daily regional average via join with the station dimension (whole history)\n        "Q3_join": """SELECT m.region, t.day, avg(t.s4), count(*) FROM t JOIN stations m\n            ON t.id_station = m.id_station GROUP BY m.region, t.day""",\n        # Q4: one station, one sensor, a 2-hour range (point-ish lookup)\n        "Q4_range": f"""SELECT time, s1 FROM t WHERE id_station = \'st1\'\n            AND day IN (\'{_day(r0)}\', \'{_day(r0 + pd.Timedelta(hours=2))}\')\n            AND time >= TIMESTAMP \'{_ts(r0)}\' AND time < TIMESTAMP \'{_ts(r0 + pd.Timedelta(hours=2))}\'""",\n        # Q5: full scan / checksum\n        "Q5_scan": "SELECT count(*) n, sum(s0) s FROM t",\n    }\n', 'run_one.py': '"""Execute one RunConfig in a fresh JVM and write results/<run_id>.json.\n\n    python -m sensorlst.run_one --config \'<RunConfig json>\' --staging DIR --workdir DIR --out DIR\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport platform\nimport shutil\nimport time\nimport traceback\nfrom datetime import datetime, timezone\n\nfrom .config import VERSIONS, RunConfig\nfrom .formats import ADAPTERS, build_spark\nfrom .queries import make_queries\n\nTT_BACK = 5  # time travel target: the version TT_BACK commits before the latest\n\n\ndef _timed(fn, *a, **kw):\n    t = time.perf_counter()\n    r = fn(*a, **kw)\n    return time.perf_counter() - t, r\n\n\nclass Ledger:\n    """Tracks bytes written per operation by diffing the table directory (nothing is cleaned mid-run)."""\n\n    def __init__(self, adapter):\n        self.a, self.seen = adapter, {}\n        self.data_written = self.meta_written = 0\n\n    def step(self) -> dict:\n        now = self.a.disk_files()\n        new_data = new_meta = 0\n        for p, s in now.items():\n            if self.seen.get(p) == s:\n                continue\n            kind = self.a.classify(p)\n            grow = s - self.seen.get(p, 0) if p in self.seen else s\n            if kind == "data":\n                new_data += max(grow, 0)\n            elif kind == "meta":\n                new_meta += max(grow, 0)\n        self.seen = now\n        self.data_written += new_data\n        self.meta_written += new_meta\n        return {"data_bytes_written": new_data, "meta_bytes_written": new_meta}\n\n    def on_disk(self) -> dict:\n        out = {"data": 0, "meta": 0, "data_files_on_disk": 0, "meta_files_on_disk": 0}\n        for p, s in self.a.disk_files().items():\n            k = self.a.classify(p)\n            if k != "ignore":\n                out[k] += s\n                out[f"{k}_files_on_disk"] += 1\n        return {"data_bytes_on_disk": out["data"], "meta_bytes_on_disk": out["meta"],\n                "data_files_on_disk": out["data_files_on_disk"], "meta_files_on_disk": out["meta_files_on_disk"]}\n\n\ndef run(cfg: RunConfig, staging: str, workdir: str, out_dir: str, cores: int, driver_mem: str, ivy: str | None) -> dict:\n    with open(os.path.join(staging, "manifest.json")) as f:\n        manifest = json.load(f)\n    shutil.rmtree(workdir, ignore_errors=True)\n    os.makedirs(workdir)\n    res = {"config": json.loads(cfg.to_json()), "run_id": cfg.run_id, "versions": VERSIONS,\n           "host": {"platform": platform.platform(), "cpus": os.cpu_count(), "cores_used": cores,\n                    "driver_mem": driver_mem},\n           "started": datetime.now(timezone.utc).isoformat(), "ops": [], "checkpoints": []}\n\n    t_jvm, spark = _timed(build_spark, cfg.fmt, workdir, cores, driver_mem, ivy)\n    res["jvm_start_s"] = t_jvm\n    res["host"]["java"] = spark.sparkContext._jvm.System.getProperty("java.version")\n    res["host"]["python"] = platform.python_version()\n    a = ADAPTERS[cfg.fmt](spark, workdir, cfg.mode)\n    spark.read.parquet(os.path.join(staging, "stations.parquet")).createOrReplaceTempView("stations")\n    led = Ledger(a)\n\n    def op(kind, batch, fn, *args):\n        dt, _ = _timed(fn, *args)\n        rec = {"op": kind, "batch": batch, "seconds": dt, **led.step()}\n        res["ops"].append(rec)\n        return rec\n\n    op("create", 0, a.create, spark.read.parquet(os.path.join(staging, "base.parquet")))\n    deletes = {}\n    for d in manifest["deletes"]:\n        deletes.setdefault(d["batch"], []).append(d)\n    checkpoints = {c["batch"]: c for c in manifest["checkpoints"]}\n\n    for b in manifest["batches"]:\n        i = b["batch"]\n        df = spark.read.parquet(os.path.join(staging, f"batch_{i:03d}.parquet"))\n        op("upsert" if cfg.u > 0 else "append", i, a.upsert if cfg.u > 0 else a.append, df)\n        for d in deletes.get(i, []):\n            op("delete", i, a.delete, d["id_station"], d["day"])\n        if cfg.compact_every and i % cfg.compact_every == 0:\n            op("compact", i, a.compact)\n        if i in checkpoints:\n            res["checkpoints"].append(checkpoint(a, spark, checkpoints[i], led))\n\n    final = res["checkpoints"][-1]\n    res["correct"] = all(c["correct"] for c in res["checkpoints"])\n    res["final"] = {k: final[k] for k in ("count", "sum_s0", "expected_count", "expected_sum_s0")}\n    res["logical_bytes"] = logical_bytes(a, workdir)\n    res["write_amplification"] = led.data_written / res["logical_bytes"] if res["logical_bytes"] else None\n    res["total_write_s"] = sum(o["seconds"] for o in res["ops"] if o["op"] in ("create", "append", "upsert", "delete"))\n    res["total_compact_s"] = sum(o["seconds"] for o in res["ops"] if o["op"] == "compact")\n    res["finished"] = datetime.now(timezone.utc).isoformat()\n    spark.stop()\n    return res\n\n\ndef checkpoint(a, spark, cp: dict, led: Ledger) -> dict:\n    view = a.read()\n    view.createOrReplaceTempView("t")\n    out = {"batch": cp["batch"], "queries": {}}\n    for name, q in make_queries(cp["event_time_end"]).items():\n        times = []\n        for _ in range(2):  # 1st = cold-ish (fresh plan/file listing), 2nd = warm\n            dt, rows = _timed(lambda: spark.sql(q).collect())\n            times.append(dt)\n        out["queries"][name] = times\n        if name == "Q5_scan":\n            out["count"], out["sum_s0"] = int(rows[0].n), round(float(rows[0].s or 0), 4)\n    out["expected_count"], out["expected_sum_s0"] = cp["expected_count"], cp["expected_sum_s0"]\n    out["correct"] = (out["count"] == cp["expected_count"]\n                      and abs(out["sum_s0"] - cp["expected_sum_s0"]) <= 1e-3 * max(1, cp["expected_count"]) ** 0.5)\n    out.update(a.live_files())\n    out.update(led.on_disk())\n    out["data_bytes_written_cum"] = led.data_written\n    out["meta_bytes_written_cum"] = led.meta_written\n    if a.supports_time_travel:\n        vs = a.versions()\n        out["n_versions"] = len(vs)\n        tt = {}\n        for label, ref in (("tt_back", vs[max(0, len(vs) - 1 - TT_BACK)]), ("tt_first", vs[0])):\n            dt, rows = _timed(lambda: a.read_version(ref).agg({"*": "count"}).collect())\n            tt[label] = {"seconds": dt, "count": int(rows[0][0]), "ref": str(ref)}\n        out["time_travel"] = tt\n    return out\n\n\ndef logical_bytes(a, workdir: str) -> int:\n    """Size of the final logical table written once as plain snappy Parquet (denominator of write amplification)."""\n    p = os.path.join(workdir, "logical")\n    cols = [c for c in a.read().columns if not c.startswith("_hoodie_") and c != "hkey"]\n    a.read().select(*cols).repartition("day").write.partitionBy("day").parquet(p)\n    total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(p) for f in fs\n                if f.endswith(".parquet"))\n    shutil.rmtree(p, ignore_errors=True)\n    return total\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", required=True)\n    ap.add_argument("--staging", required=True)\n    ap.add_argument("--workdir", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--cores", type=int, default=4)\n    ap.add_argument("--driver-mem", default="8g")\n    ap.add_argument("--ivy", default=None)\n    ap.add_argument("--keep", action="store_true", help="keep the table directory (for demos)")\n    args = ap.parse_args()\n    cfg = RunConfig.from_json(args.config)\n    os.makedirs(args.out, exist_ok=True)\n    try:\n        res = run(cfg, args.staging, args.workdir, args.out, args.cores, args.driver_mem, args.ivy)\n    except Exception:\n        with open(os.path.join(args.out, "failures.jsonl"), "a") as f:\n            f.write(json.dumps({"run_id": cfg.run_id, "error": traceback.format_exc()[-4000:]}) + "\\n")\n        raise\n    finally:\n        if not args.keep:\n            shutil.rmtree(args.workdir, ignore_errors=True)\n    with open(os.path.join(args.out, f"{cfg.run_id}.json"), "w") as f:\n        json.dump(res, f, indent=1)\n    print(f"{cfg.run_id}: correct={res[\'correct\']} write={res[\'total_write_s\']:.1f}s "\n          f"compact={res[\'total_compact_s\']:.1f}s WA={res[\'write_amplification\']:.2f}")\n\n\nif __name__ == "__main__":\n    main()\n', 'workload.py': '"""sensor-LST workload generator.\n\nTurns a TSM-Bench-shaped source (time, id_station, s0..sK) into a deterministic stream of\noperations staged as Parquet files, so every table format ingests byte-identical input:\n\n  base.parquet          history bulk-loaded at t0\n  batch_NNN.parquet     micro-batch NNN = new rows for its event-time slice\n                        + round(u * rows_per_batch) late corrections of existing keys\n  stations.parquet      station metadata (dimension table for join queries)\n  manifest.json         deletes, checkpoints and the expected table state (ground truth)\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nfrom datetime import datetime, timedelta, timezone\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom .config import SCALES, Scale\n\nKEY = ["id_station", "time"]\n\n\ndef sensor_cols(scale: Scale) -> list[str]:\n    return [f"s{i}" for i in range(scale.n_sensors)]\n\n\ndef _t0(scale: Scale) -> pd.Timestamp:\n    return pd.Timestamp(scale.start, tz="UTC")\n\n\ndef horizon_hours(scale: Scale) -> float:\n    return scale.base_hours + scale.n_batches * scale.batch_minutes / 60\n\n\ndef synthetic_source(scale: Scale, seed: int = 0) -> pd.DataFrame:\n    """Same shape and value distribution as TSM-Bench d1 (uniform [0,1), 6 decimals). For tests/smoke runs."""\n    rng = np.random.default_rng(seed)\n    n_ts = int(horizon_hours(scale) * 3600 // scale.interval_s)\n    times = _t0(scale) + pd.to_timedelta(np.arange(n_ts) * scale.interval_s, unit="s")\n    frames = []\n    for st in range(scale.n_stations):\n        df = pd.DataFrame({"time": times, "id_station": f"st{st}"})\n        vals = rng.random((n_ts, scale.n_sensors)).round(6)\n        frames.append(pd.concat([df, pd.DataFrame(vals, columns=sensor_cols(scale))], axis=1))\n    return pd.concat(frames, ignore_index=True)\n\n\ndef prepare_d1(split_dir: str, out_path: str, scale: Scale, chunksize: int = 500_000) -> pd.DataFrame:\n    """Stream TSM-Bench d1 (split tar.gz -> d1.csv, 4.8 GB) and keep only what `scale` needs.\n\n    d1 is sorted by station then time, so the whole file is scanned; nothing is extracted to disk.\n    """\n    parts = sorted(os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.startswith("datasets_splits."))\n    stations = {f"st{i}" for i in range(scale.n_stations)}\n    end = _t0(scale) + pd.Timedelta(hours=horizon_hours(scale))\n    cat = subprocess.Popen(["cat", *parts], stdout=subprocess.PIPE)\n    tar = subprocess.Popen(["tar", "-xzOf", "-", "d1.csv"], stdin=cat.stdout, stdout=subprocess.PIPE)\n    cat.stdout.close()\n    keep = []\n    for chunk in pd.read_csv(tar.stdout, usecols=["time", "id_station", *sensor_cols(scale)], chunksize=chunksize):\n        chunk = chunk[chunk.id_station.isin(stations)]\n        if chunk.empty:\n            continue\n        chunk["time"] = pd.to_datetime(chunk["time"], utc=True)\n        keep.append(chunk[(chunk.time >= _t0(scale)) & (chunk.time < end)])\n    tar.wait()\n    df = pd.concat(keep, ignore_index=True).drop_duplicates(KEY).sort_values(KEY, ignore_index=True)\n    _write(df, out_path)\n    return df\n\n\ndef _write(df: pd.DataFrame, path: str) -> None:\n    table = pa.Table.from_pandas(df, preserve_index=False)\n    pq.write_table(table, path, coerce_timestamps="us", allow_truncated_timestamps=True, compression="snappy")\n\n\ndef _with_cols(df: pd.DataFrame, batch: int, scols: list[str]) -> pd.DataFrame:\n    out = df[["time", "id_station", *scols]].copy()\n    out["day"] = out["time"].dt.strftime("%Y-%m-%d")\n    out["ingest_batch"] = np.int32(batch)\n    return out[["time", "id_station", "day", "ingest_batch", *scols]]\n\n\ndef build_plan(scale_name: str, u: float, out_dir: str, source: pd.DataFrame | None = None, seed: int = 42) -> dict:\n    """Stage the operation stream for (scale, u) and return the manifest. Deterministic in (source, u, seed)."""\n    scale = SCALES[scale_name]\n    scols = sensor_cols(scale)\n    if source is None:\n        source = synthetic_source(scale, seed=0)\n    source = source[source.id_station.isin([f"st{i}" for i in range(scale.n_stations)])]\n    os.makedirs(out_dir, exist_ok=True)\n    rng = np.random.default_rng([seed, int(round(u * 10_000))])\n\n    t0 = _t0(scale)\n    base_end = t0 + pd.Timedelta(hours=scale.base_hours)\n    step = pd.Timedelta(minutes=scale.batch_minutes)\n    late = pd.Timedelta(hours=scale.late_window_hours)\n\n    base = _with_cols(source[source.time < base_end], 0, scols)\n    _write(base, os.path.join(out_dir, "base.parquet"))\n\n    # Ground-truth state: key -> (time, s0). Only s0 is tracked; it is enough to checksum updates.\n    state = base[["id_station", "time", "day", "s0"]].set_index(KEY)\n    n_late = int(round(u * scale.rows_per_batch()))\n    deletes, checkpoints, batches = [], [], []\n    stations = sorted(source.id_station.unique(), key=lambda s: int(s[2:]))\n\n    for i in range(1, scale.n_batches + 1):\n        b_start = base_end + (i - 1) * step\n        b_end = b_start + step\n        new = source[(source.time >= b_start) & (source.time < b_end)]\n        parts = [_with_cols(new, i, scols)]\n        n_upd = 0\n        if n_late:\n            window = state[(state.index.get_level_values("time") >= b_start - late)]\n            pick = rng.choice(len(window), size=min(n_late, len(window)), replace=False)\n            keys = window.index[np.sort(pick)]\n            corr = pd.DataFrame({"id_station": keys.get_level_values(0), "time": keys.get_level_values(1)})\n            corr[scols] = rng.random((len(corr), len(scols))).round(6)\n            parts.append(_with_cols(corr, i, scols))\n            state.loc[keys, "s0"] = corr["s0"].to_numpy()\n            n_upd = len(corr)\n        batch = pd.concat(parts, ignore_index=True)\n        _write(batch, os.path.join(out_dir, f"batch_{i:03d}.parquet"))\n        state = pd.concat([state, _with_cols(new, i, scols)[["id_station", "time", "day", "s0"]].set_index(KEY)])\n        batches.append({"batch": i, "rows": len(batch), "new_rows": len(new), "late_rows": n_upd,\n                        "days": sorted(batch.day.unique().tolist())})\n\n        if i % scale.delete_every == 0:\n            k = i // scale.delete_every - 1\n            station = stations[k % len(stations)]\n            day = (t0 + pd.Timedelta(days=k % max(1, int(scale.base_hours // 24)))).strftime("%Y-%m-%d")\n            mask = (state.index.get_level_values("id_station") == station) & (state.day == day)\n            deletes.append({"batch": i, "id_station": station, "day": day, "rows": int(mask.sum())})\n            state = state[~mask]\n\n        if i % scale.checkpoint_every == 0 or i == scale.n_batches:\n            checkpoints.append({"batch": i, "event_time_end": b_end.isoformat(),\n                                "expected_count": int(len(state)),\n                                "expected_sum_s0": round(float(state.s0.sum()), 4)})\n\n    stations_df = pd.DataFrame({"id_station": stations,\n                                "region": [f"r{int(s[2:]) % 3}" for s in stations],\n                                "elevation_m": [100 + 37 * int(s[2:]) for s in stations]})\n    stations_df.to_parquet(os.path.join(out_dir, "stations.parquet"), index=False)\n\n    manifest = {"scale": scale_name, "u": u, "seed": seed, "sensors": scols, "base_rows": len(base),\n                "base_end": base_end.isoformat(), "batches": batches, "deletes": deletes,\n                "checkpoints": checkpoints, "created": datetime.now(timezone.utc).isoformat()}\n    with open(os.path.join(out_dir, "manifest.json"), "w") as f:\n        json.dump(manifest, f, indent=1)\n    return manifest\n'}
PKG = os.path.join(WORK, 'sensorlst')
os.makedirs(PKG, exist_ok=True)
for name, src in FILES.items():
    with open(os.path.join(PKG, name), 'w', encoding='utf-8') as f:
        f.write(src)
print(sorted(os.listdir(PKG)))

## Data: TSM-Bench d1 → Parquet subset (10 stations × 20 sensors × 17 days)

In [ ]:
SRC = f"{SCRATCH}/d1_{SCALE}.parquet"
# Optional shortcut: a Kaggle Dataset holding d1_small.parquet (built by sensorlst.workload.prepare_d1) is used as-is.
attached = glob.glob(f"/kaggle/input/**/d1_{SCALE}.parquet", recursive=True)
if attached and not os.path.exists(SRC):
    shutil.copy(attached[0], SRC)
    print("using attached dataset", attached[0])
if not os.path.exists(SRC):
    t = time.time()
    repo = f"{SCRATCH}/tsmbench"
    for attempt in range(1, 4):
        shutil.rmtree(repo, ignore_errors=True)
        r = subprocess.run(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/eXascaleInfolab/TSM-Bench.git {repo}"
                           f" && cd {repo} && git sparse-checkout set datasets", shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            break
        print(f"clone attempt {attempt} failed: {r.stderr[-300:]}"); time.sleep(30)
    else:
        raise RuntimeError("Cannot download TSM-Bench from GitHub. Upload d1_small.parquet as a Kaggle Dataset, "
                           "attach it (Add Input), and run again.")
    prep = ("import sys; from sensorlst.config import SCALES; from sensorlst.workload import prepare_d1; "
            "df = prepare_d1(sys.argv[1], sys.argv[2], SCALES[sys.argv[3]]); print(df.shape, df.time.min(), df.time.max())")
    r = subprocess.run([PY, "-c", prep, f"{repo}/datasets/d1_splits", SRC, SCALE], cwd=WORK, capture_output=True, text=True)
    print(r.stdout, r.stderr[-3000:] if r.returncode else "", f"{time.time() - t:.0f}s")
    assert r.returncode == 0 and os.path.exists(SRC), "data preparation failed (see error above)"
    shutil.rmtree(repo, ignore_errors=True)

## Warm the Maven/Ivy cache so package download is not timed as JVM start

In [ ]:
warm = '''
import sys
from sensorlst.formats import build_spark
s = build_spark(sys.argv[1], "/tmp/lh/warm", 1, "1g", "/tmp/lh/ivy"); s.range(1).count(); s.stop()
'''
for fmt in ["delta", "iceberg", "hudi"]:
    r = subprocess.run([PY, "-c", warm, fmt], cwd=WORK, capture_output=True, text=True)
    print(fmt, "ok" if r.returncode == 0 else r.stderr[-1500:])

## Run the shard (resumable, prints one line per run)

In [ ]:
TAG = f"u{int(round(U * 100)):02d}"
OUT = f"{RESULTS}/runs_{TAG}"
cmd = [PY, "-m", "sensorlst.matrix", "--scale", SCALE, "--u", str(U), "--reps", str(REPS),
       "--source", SRC, "--out", OUT, "--staging", f"{SCRATCH}/staging", "--workdir", f"{SCRATCH}/work",
       "--cores", str(CORES), "--driver-mem", DRIVER_MEM, "--ivy", f"{SCRATCH}/ivy",
       "--budget-hours", str(BUDGET_HOURS), "--calibration"]
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print("exit", p.returncode, "runs:", len(glob.glob(f"{OUT}/*.json")))

## Shard summary + package results

In [ ]:
# Self-contained: also works on its own after a kernel restart.
import os, sys, glob, shutil, subprocess
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
RESULTS = next((d for d in ["/content/drive/MyDrive/sensorlst/results", f"{WORK}/results"]
                if glob.glob(f"{d}/runs_*/*.json")), f"{WORK}/results")
runs = sorted(glob.glob(f"{RESULTS}/runs_*"))
print("result folders:", {os.path.basename(r): len(glob.glob(f"{r}/*.json")) for r in runs})
for r in runs:
    tag = os.path.basename(r).replace("runs_", "")
    p = subprocess.run([sys.executable, "-m", "sensorlst.analyze", r, "--out", f"{RESULTS}/report_{tag}"],
                       cwd=WORK, capture_output=True, text=True)
    print(p.stdout or p.stderr[-2000:])
    if os.path.exists(f"{RESULTS}/report_{tag}/summary.md"):
        print(open(f"{RESULTS}/report_{tag}/summary.md").read()[:3000])
    zipname = shutil.make_archive(f"{WORK}/results_{tag}", "zip", RESULTS)
    print("done:", zipname)
if not runs:
    print("No finished runs found yet: the 'Run the shard' cell has not produced results in this session.")